<a href="https://colab.research.google.com/github/MiraM29/learningsql-2875059/blob/main/Burns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
from google.colab import files
uploaded = files.upload()  # This opens a file picker in Colab

Saving Promotion_chain_initial.xlsx to Promotion_chain_initial.xlsx


In [42]:
import pandas as pd

# Get the filename from the uploaded dictionary
# Assuming only one file is uploaded, or the first one is desired
filename = next(iter(uploaded))

# Read the Excel file using the dynamically obtained filename, specifying the second sheet (index 1)
df = pd.read_excel(filename, sheet_name=1)

# Check it loaded
display(df)

,source,target,sign
0,q1,q2,1
1,q2,q8,1
2,q2,q17,1
3,q3,q2,-1
4,q3,q4,1
5,q4,q11,1
6,q4,q17,1
7,q5,q4,-1
8,q5,q6,1
9,q6,q15,1


In [43]:
!pip install openpyxl # read and write excel

In [44]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import (
    Any,
    Callable,
    Dict,
    FrozenSet,
    Hashable,
    Iterable,
    List,
    Optional,
    Set,
    Tuple,
)

Quantity = Hashable
CouplingId = Hashable


class BurnsAlgorithmError(RuntimeError):
    pass


@dataclass(frozen=True)
class Coupling:
    """Directed coupling cij from src -> dst."""
    cid: CouplingId
    src: Quantity
    dst: Quantity


@dataclass
class SignedGraph:
    """D = {Q, C} with directed couplings."""
    quantities: FrozenSet[Quantity]
    couplings: FrozenSet[Coupling]
    _by_id: Dict[CouplingId, Coupling] = field(default_factory=dict, init=False, repr=False)
    _out: Dict[Quantity, Set[CouplingId]] = field(default_factory=dict, init=False, repr=False)
    _in: Dict[Quantity, Set[CouplingId]] = field(default_factory=dict, init=False, repr=False)
    _pairs: Set[Tuple[Quantity, Quantity]] = field(default_factory=set, init=False, repr=False)

    def __post_init__(self) -> None:
        object.__setattr__(self, "_by_id", {c.cid: c for c in self.couplings})

        out: Dict[Quantity, Set[CouplingId]] = {q: set() for q in self.quantities}
        inn: Dict[Quantity, Set[CouplingId]] = {q: set() for q in self.quantities}
        pairs: Set[Tuple[Quantity, Quantity]] = set()

        for c in self.couplings:
            if c.src not in out or c.dst not in inn:
                raise ValueError(f"Coupling endpoints must be in quantities. Bad coupling: {c}")
            out[c.src].add(c.cid)
            inn[c.dst].add(c.cid)
            pairs.add((c.src, c.dst))

        object.__setattr__(self, "_out", out)
        object.__setattr__(self, "_in", inn)
        object.__setattr__(self, "_pairs", pairs)

    @staticmethod
    def from_edges(
        quantities: Iterable[Quantity],
        edges: Iterable[Tuple[Quantity, Quantity, Optional[CouplingId]]],
    ) -> "SignedGraph":
        qset = frozenset(quantities)
        couplings: List[Coupling] = []
        used: Set[CouplingId] = set()

        for (src, dst, cid) in edges:
            if cid is None:
                cid = f"c{src}->{dst}"
            if cid in used:
                raise ValueError(f"Duplicate coupling id: {cid}")
            used.add(cid)
            couplings.append(Coupling(cid=cid, src=src, dst=dst))

        return SignedGraph(quantities=qset, couplings=frozenset(couplings))

    def coupling(self, cid: CouplingId) -> Coupling:
        return self._by_id[cid]

    def Ec(self, q: Quantity) -> Set[CouplingId]:
        """Effector couplings: outgoing from q."""
        return set(self._out.get(q, set()))

    def Ac(self, q: Quantity) -> Set[CouplingId]:
        """Affector couplings: incoming to q."""
        return set(self._in.get(q, set()))

    def has_pair(self, a: Quantity, b: Quantity) -> bool:
        return (a, b) in self._pairs

    def reverse_couplings(self, a: Quantity, b: Quantity) -> Set[CouplingId]:
        """All coupling IDs for b -> a (reverse direction)."""
        return {cid for cid in self._out.get(b, set()) if self._by_id[cid].dst == a}


@dataclass
class BurnsState:
    # Step 1 sets
    Qun: Set[Quantity] = field(default_factory=set)
    Cun: Set[CouplingId] = field(default_factory=set)

    PU: Set[Quantity] = field(default_factory=set)
    RV: Set[Quantity] = field(default_factory=set)
    VX: Set[Quantity] = field(default_factory=set)

    F: Set[CouplingId] = field(default_factory=set)
    I: Set[CouplingId] = field(default_factory=set)

    P: Set[Quantity] = field(default_factory=set)
    R: Set[Quantity] = field(default_factory=set)
    U: Set[Quantity] = field(default_factory=set)
    V: Set[Quantity] = field(default_factory=set)
    X: Set[Quantity] = field(default_factory=set)
    Y: Set[Quantity] = field(default_factory=set)

    def snapshot(self, step: int, note: str = "", iteration: int = 0) -> Dict[str, Any]:
        def srt(xs: Set[Any]) -> List[str]:
            return sorted((str(x) for x in xs), key=lambda z: z)

        return {
            "step": step,
            "iteration": iteration,
            "note": note,
            "Qun": srt(self.Qun),
            "Cun": srt(self.Cun),
            "PU": srt(self.PU),
            "RV": srt(self.RV),
            "VX": srt(self.VX),
            "F": srt(self.F),
            "I": srt(self.I),
            "P": srt(self.P),
            "R": srt(self.R),
            "U": srt(self.U),
            "V": srt(self.V),
            "X": srt(self.X),
            "Y": srt(self.Y),
        }


Decision = str  # "state" | "rate" | "aux"


def run_burns_algorithm(
    g: SignedGraph,
    *,
    decisions_step8: Optional[Dict[Quantity, Decision]] = None,
    controllable: Optional[Callable[[Quantity], bool]] = None,
    infer_connectors_from_quantity_type: bool = True,
    max_major_iterations: int = 200,
) -> List[Dict[str, Any]]:
    """
    Implements Steps 1-9, yielding snapshots after each step execution.

    - decisions_step8: mapping for Step 8 if Qun cannot be fully resolved automatically.
      Allowed values: "state", "rate", "aux"
    - controllable(q): used in Step 9 to split PU into P and U. If None => all go to P.
    - infer_connectors_from_quantity_type: applies Burns (2001) summarized rules once a quantity is typed.
    """
    if decisions_step8 is None:
        decisions_step8 = {}

    def is_controllable(q: Quantity) -> bool:
        if controllable is None:
            return False
        return bool(controllable(q))

    st = BurnsState()
    snapshots: List[Dict[str, Any]] = []

    # -------------------------
    # Step 1: Initialization
    # -------------------------
    st.Qun = set(g.quantities)
    st.Cun = {c.cid for c in g.couplings}
    # PU=RV=VX=F=I=P=R=U=V=X=Y=∅ already by default
    snapshots.append(st.snapshot(1, "Initialized sets"))

    def move_couplings_to_I(cids: Set[CouplingId]) -> None:
        cids2 = set(cids) & st.Cun
        st.I |= cids2
        st.Cun -= cids2

    def move_couplings_to_F(cids: Set[CouplingId]) -> None:
        cids2 = set(cids) & st.Cun
        st.F |= cids2
        st.Cun -= cids2

    def infer_from_quantity_types() -> None:
        """
        Optional: Burns (2001) summarized rules about connector subsets once q is identified:
        - q in R: incoming ⊆ I, outgoing ⊆ F
        - q in X: incoming ⊆ F, outgoing ⊆ I
        - q in V: incoming ⊆ I, outgoing ⊆ I
        - q in PU: incoming = ∅, outgoing ⊆ I (we only enforce outgoing ⊆ I)
        - q in Y: incoming ⊆ I, outgoing = ∅ (we only enforce incoming ⊆ I)
        """
        if not infer_connectors_from_quantity_type:
            return

        # Rates
        for q in st.R:
            move_couplings_to_I(g.Ac(q))
            move_couplings_to_F(g.Ec(q))

        # States
        for q in st.X:
            move_couplings_to_F(g.Ac(q))
            move_couplings_to_I(g.Ec(q))

        # Auxiliaries
        for q in st.V:
            move_couplings_to_I(g.Ac(q))
            move_couplings_to_I(g.Ec(q))

        # Inputs/parameters
        for q in st.PU:
            move_couplings_to_I(g.Ec(q))

        # Outputs
        for q in st.Y:
            move_couplings_to_I(g.Ac(q))

    # -------------------------
    # Step 2: Classify nonendogenous quantities + associated couplings
    # -------------------------
    qlist = list(st.Qun)
    for qi in qlist:
        if len(g.Ec(qi)) == 0:
            st.Y.add(qi)
            st.Qun.discard(qi)
        if len(g.Ac(qi)) == 0:
            st.PU.add(qi)
            st.Qun.discard(qi)

    move_couplings_to_I(set().union(*(g.Ac(q) for q in st.Y)) if st.Y else set())
    move_couplings_to_I(set().union(*(g.Ec(q) for q in st.PU)) if st.PU else set())

    infer_from_quantity_types()
    snapshots.append(st.snapshot(2, "Classified Y and PU; moved Ac(Y), Ec(PU) to I"))

    def step5_classify() -> Tuple[bool, bool]:
        """Returns (new_RV_added, new_VX_added)."""
        new_rv = False
        new_vx = False

        for qi in list(st.Qun):
            Ac_q = g.Ac(qi)
            Ec_q = g.Ec(qi)

            if Ac_q.issubset(st.I) and Ec_q.issubset(st.I):
                st.V.add(qi)
                st.Qun.discard(qi)
                st.RV.discard(qi)
                st.VX.discard(qi)
            else:
                if Ac_q.issubset(st.I):
                    if qi not in st.RV:
                        new_rv = True
                    st.RV.add(qi)
                if Ec_q.issubset(st.I):
                    if qi not in st.VX:
                        new_vx = True
                    st.VX.add(qi)

        return new_rv, new_vx

    # -------------------------
    # Main loop: Steps 3..7 repeat
    # -------------------------
    major_iter = 0
    while True:
        major_iter += 1
        if major_iter > max_major_iterations:
            raise BurnsAlgorithmError("Max iterations exceeded (graph may be ambiguous).")

        # -------------------------
        # Step 3: Apply consistency supposition
        # -------------------------
        changed = True
        while changed:
            changed = False

            for cid in list(st.I):
                c = g.coupling(cid)
                add1 = (st.Cun & g.Ec(c.src))
                add2 = (st.Cun & g.Ac(c.dst))
                if add1 or add2:
                    before = len(st.Cun)
                    move_couplings_to_I(add1 | add2)
                    changed = changed or (len(st.Cun) != before)

            for cid in list(st.F):
                c = g.coupling(cid)
                add1 = (st.Cun & g.Ec(c.src))
                add2 = (st.Cun & g.Ac(c.dst))
                if add1 or add2:
                    before = len(st.Cun)
                    move_couplings_to_F(add1 | add2)
                    changed = changed or (len(st.Cun) != before)

        infer_from_quantity_types()
        snapshots.append(st.snapshot(3, "Applied consistency supposition", iteration=major_iter))

        # -------------------------
        # Step 4: Classify R and X from consideration of F
        # -------------------------
        for qi in list(st.Qun):
            if g.Ac(qi).issubset(st.F):
                st.X.add(qi)
                st.Qun.discard(qi)
                st.VX.discard(qi)
            elif g.Ec(qi).issubset(st.F):
                st.R.add(qi)
                st.Qun.discard(qi)
                st.RV.discard(qi)

        infer_from_quantity_types()
        snapshots.append(st.snapshot(4, "Classified X and R using F", iteration=major_iter))

        # -------------------------
        # Step 5: Classify V, RV, VX from consideration of I
        # -------------------------
        new_rv, new_vx = step5_classify()
        infer_from_quantity_types()
        snapshots.append(st.snapshot(5, "Classified V / updated RV and VX using I", iteration=major_iter))

        if len(st.Qun) == 0:
            break
        if (not new_rv) and (not new_vx):
            # -------------------------
            # Step 8: Specify final classification of a member of Qun
            # -------------------------
            qi = min(st.Qun, key=lambda x: str(x))
            decision = decisions_step8.get(qi)

            if decision not in {"state", "rate", "aux"}:
                raise BurnsAlgorithmError(
                    "Step 8 required but no decision provided. "
                    f"Need decisions_step8[{qi!r}] in {{'state','rate','aux'}}. "
                    f"Remaining Qun={sorted(map(str, st.Qun))}"
                )

            if decision == "state":
                st.X.add(qi)
                st.Qun.discard(qi)
                st.VX.discard(qi)
                move_couplings_to_I(g.Ec(qi) & st.Cun)
                move_couplings_to_F(g.Ac(qi) & st.Cun)

            elif decision == "rate":
                st.R.add(qi)
                st.Qun.discard(qi)
                st.RV.discard(qi)
                move_couplings_to_F(g.Ec(qi) & st.Cun)
                move_couplings_to_I(g.Ac(qi) & st.Cun)

            else:  # aux
                st.V.add(qi)
                st.Qun.discard(qi)
                st.RV.discard(qi)
                st.VX.discard(qi)
                move_couplings_to_I(g.Ec(qi) & st.Cun)
                move_couplings_to_I(g.Ac(qi) & st.Cun)

            infer_from_quantity_types()
            snapshots.append(st.snapshot(8, f"Step 8 manual classification: {qi} -> {decision}", iteration=major_iter))

            if len(st.Qun) == 0:
                break
            else:
                continue

        # -------------------------
        # Step 6: Search for minor submodels containing members of VX
        # -------------------------
        for qi in list(st.VX):
            out_cids = g.Ec(qi)
            for cid in out_cids:
                c = g.coupling(cid)
                qj = c.dst
                if g.has_pair(qj, qi):
                    st.X.add(qi)
                    st.Qun.discard(qi)
                    st.VX.discard(qi)

                    st.R.add(qj)
                    st.Qun.discard(qj)
                    st.RV.discard(qj)

                    rev = g.reverse_couplings(a=qi, b=qj)
                    move_couplings_to_F(rev)

        infer_from_quantity_types()
        snapshots.append(st.snapshot(6, "Searched minor submodels (VX)", iteration=major_iter))

        # -------------------------
        # Step 7: Consider cardinality of Ec(qi) for qi in RV
        # -------------------------
        for qi in list(st.RV):
            if len(g.Ec(qi)) > 2:
                st.V.add(qi)
                st.RV.discard(qi)
                st.Qun.discard(qi)
                move_couplings_to_I(g.Ec(qi))

        infer_from_quantity_types()
        snapshots.append(st.snapshot(7, "Applied RV effector-cardinality rule", iteration=major_iter))

        if len(st.Qun) == 0:
            break

    # -------------------------
    # Step 9: Partition PU into P and U
    # -------------------------
    for qi in st.PU:
        if is_controllable(qi):
            st.U.add(qi)
        else:
            st.P.add(qi)

    snapshots.append(st.snapshot(9, "Partitioned PU into P and U"))
    return snapshots



    snaps = run_burns_algorithm(
        g,
        decisions_step8={"qB": "aux"},   # only needed if Step 8 is reached
        controllable=lambda q: q == "qA" # for Step 9
    )

    for s in snaps:
        print(f"\n--- Step {s['step']} (iter={s['iteration']}) {s['note']} ---")
        print("Qun:", s["Qun"])
        print("PU:", s["PU"], "P:", s["P"], "U:", s["U"])
        print("R:", s["R"], "X:", s["X"], "V:", s["V"], "Y:", s["Y"])
        print("I:", s["I"], "F:", s["F"], "Cun:", s["Cun"])


In [45]:
import pandas as pd
from IPython.display import Markdown, display

# Assuming SignedGraph and run_burns_algorithm are defined in a previous cell and available
# (Cell f-q4IkP9ypi4 defines these)

# --- Graph Creation and Algorithm Execution ---
# Prepare data for SignedGraph from the 'df' DataFrame
# 'df' is available from kernel state, likely loaded from the excel file.
quantities_from_df = list(pd.concat([df['source'], df['target']]).unique())
edges_from_df = []
for _, row in df.iterrows():
    # Creating a unique coupling ID. The 'sign' column is in the df, but the BurnsAlgorithm
    # as implemented here (SignedGraph class) does not directly use it for classification logic
    # beyond defining directed couplings.
    # The current SignedGraph and run_burns_algorithm do not use 'sign' for their classification logic.
    edges_from_df.append((row['source'], row['target'], f"c_{row['source']}_{row['target']}"))

# Create the SignedGraph
graph_from_df = SignedGraph.from_edges(
    quantities=quantities_from_df,
    edges=edges_from_df
)

# Run the Burns algorithm
# decision_step8 and controllable can be empty or None if not specifically needed for the input graph
# For now, we use defaults.
# The error indicates that decisions are needed for 'q14', 'q3', 'q4', 'q7', 'q8'
# Adding a decision for 'q14' as an example, and now for 'q7' and 'q8'.
# The previous error indicated missing decisions for 'q2', 'q3', and 'q5'.
# Added decisions for 'q2', 'q3', and 'q5' as 'aux' for demonstration. You may need to adjust these based on your model's logic.
all_snapshots = run_burns_algorithm(graph_from_df, decisions_step8={'q14': 'aux', 'q7': 'aux', 'q8': 'aux', 'q2': 'aux', 'q3': 'aux', 'q5': 'aux'}, controllable=None)


def get_incremental_changes(prev_snap, curr_snap):
    changes = {'quantity_sets': {}, 'coupling_sets': {}}

    # Define keys for quantity sets and coupling sets as found in BurnsState.snapshot output
    quantity_keys = ['Qun', 'PU', 'RV', 'VX', 'P', 'R', 'U', 'V', 'X', 'Y']
    coupling_keys = ['Cun', 'F', 'I']

    for key in quantity_keys:
        prev_set = set(prev_snap.get(key, []))
        curr_set = set(curr_snap.get(key, []))

        added = sorted(list(curr_set - prev_set))
        removed = sorted(list(prev_set - curr_set))

        if added or removed:
            changes['quantity_sets'][key] = {'added': added, 'removed': removed}

    for key in coupling_keys:
        prev_set = set(prev_snap.get(key, []))
        curr_set = set(curr_snap.get(key, []))

        added = sorted(list(curr_set - prev_set))
        removed = sorted(list(prev_set - curr_set))

        if added or removed:
            changes['coupling_sets'][key] = {'added': added, 'removed': removed}
    return changes


print("--- Incremental Changes at Each Algorithm Step ---")
print("------------------------------------------------------------------------------------------------------------------------------------------")

# Display the initial state (Step 1 Initialization)
initial_snapshot = all_snapshots[0]
display(Markdown(f"### Step {initial_snapshot['step']} (iter={initial_snapshot['iteration']}) {initial_snapshot['note']}"))
print("Quantities (Initial State):")
# Iterate over known quantity keys
for k in ['Qun', 'PU', 'RV', 'VX', 'P', 'R', 'U', 'V', 'X', 'Y']:
    if initial_snapshot.get(k): # Only print if not empty
        print(f"  {k}: {initial_snapshot.get(k, [])}")
print("\nCouplings (Initial State):")
# Iterate over known coupling keys
for k in ['Cun', 'F', 'I']:
    if initial_snapshot.get(k): # Only print if not empty
        print(f"  {k}: {initial_snapshot.get(k, [])}")
print("------------------------------------------------------------------------------------------------------------------------------------------")

prev_snapshot = initial_snapshot
for i in range(1, len(all_snapshots)):
    current_snapshot = all_snapshots[i]
    changes = get_incremental_changes(prev_snapshot, current_snapshot)

    display(Markdown(f"### Step {current_snapshot['step']} (iter={current_snapshot['iteration']}) {current_snapshot['note']}"))

    has_changes = False

    if changes['quantity_sets']:
        print("Quantities Changes:")
        for q_set, diff in changes['quantity_sets'].items():
            if diff['added']:
                print(f"  Added to {q_set}: {diff['added']}")
                has_changes = True
            if diff['removed']:
                print(f"  Removed from {q_set}: {diff['removed']}")
                has_changes = True

    if changes['coupling_sets']:
        print("\nCouplings Changes:")
        for c_set, diff in changes['coupling_sets'].items():
            if diff['added']:
                print(f"  Added to {c_set}: {diff['added']}")
                has_changes = True
            if diff['removed']:
                print(f"  Removed from {c_set}: {diff['removed']}")
                has_changes = True

    if not has_changes:
        print("  No significant changes in quantities or couplings for this step.")

    print("------------------------------------------------------------------------------------------------------------------------------------------")
    prev_snapshot = current_snapshot


# --- Final Result Table ---
final_snapshot = all_snapshots[-1]

# Prepare data for the DataFrame directly from final_snapshot
classified_data = {
    'P (Parameters)': final_snapshot.get('P', []),
    'R (Rates)': final_snapshot.get('R', []),
    'U (Inputs/Controllable)': final_snapshot.get('U', []),
    'V (Auxiliaries)': final_snapshot.get('V', []),
    'X (States)': final_snapshot.get('X', []),
    'Y (Outputs)': final_snapshot.get('Y', []),
    'Qun (Unclassified)': final_snapshot.get('Qun', []) # Should be empty at the end if fully classified
}

# Find the maximum length to align lists in the DataFrame
max_len = max([len(nodes) for nodes in classified_data.values()] or [0]) # Handle case where all lists are empty

# Pad lists with None to ensure all are same length for DataFrame creation
for key, value in classified_data.items():
    classified_data[key] = value + [None] * (max_len - len(value))

final_df = pd.DataFrame(classified_data)

display(Markdown("## Final Classification Table"))
display(final_df)

--- Incremental Changes at Each Algorithm Step ---
------------------------------------------------------------------------------------------------------------------------------------------


### Step 1 (iter=0) Initialized sets

Quantities (Initial State):
  Qun: ['a8', 'q1', 'q10', 'q11', 'q12', 'q13', 'q14', 'q15', 'q16', 'q17', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9']

Couplings (Initial State):
  Cun: ['c_q10_q3', 'c_q10_q7', 'c_q11_q2', 'c_q12_q5', 'c_q12_q7', 'c_q13_a8', 'c_q13_q12', 'c_q14_q11', 'c_q14_q5', 'c_q15_q6', 'c_q16_q15', 'c_q1_q2', 'c_q2_q17', 'c_q2_q8', 'c_q3_q2', 'c_q3_q4', 'c_q4_q11', 'c_q4_q17', 'c_q5_q4', 'c_q5_q6', 'c_q6_q15', 'c_q6_q17', 'c_q7_q2', 'c_q8_q3', 'c_q8_q7', 'c_q9_q8']
------------------------------------------------------------------------------------------------------------------------------------------


### Step 2 (iter=0) Classified Y and PU; moved Ac(Y), Ec(PU) to I

Quantities Changes:
  Removed from Qun: ['a8', 'q1', 'q10', 'q13', 'q14', 'q16', 'q17', 'q9']
  Added to PU: ['q1', 'q10', 'q13', 'q14', 'q16', 'q9']
  Added to Y: ['a8', 'q17']

Couplings Changes:
  Removed from Cun: ['c_q10_q3', 'c_q10_q7', 'c_q13_a8', 'c_q13_q12', 'c_q14_q11', 'c_q14_q5', 'c_q16_q15', 'c_q1_q2', 'c_q2_q17', 'c_q4_q17', 'c_q6_q17', 'c_q9_q8']
  Added to I: ['c_q10_q3', 'c_q10_q7', 'c_q13_a8', 'c_q13_q12', 'c_q14_q11', 'c_q14_q5', 'c_q16_q15', 'c_q1_q2', 'c_q2_q17', 'c_q4_q17', 'c_q6_q17', 'c_q9_q8']
------------------------------------------------------------------------------------------------------------------------------------------


...

Quantities Changes:
  Removed from Qun: ['q11', 'q12', 'q15', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8']
  Added to V: ['q11', 'q12', 'q15', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8']
------------------------------------------------------------------------------------------------------------------------------------------


### Step 9 (iter=0) Partitioned PU into P and U

Quantities Changes:
  Added to P: ['q1', 'q10', 'q13', 'q14', 'q16', 'q9']
------------------------------------------------------------------------------------------------------------------------------------------


## Final Classification Table

,P (Parameters),R (Rates),U (Inputs/Controllable),V (Auxiliaries),X (States),Y (Outputs),Qun (Unclassified)
0,q1,None,None,q11,None,a8,None
1,q10,None,None,q12,None,q17,None
2,q13,None,None,q15,None,None,None
3,q14,None,None,q2,None,None,None
4,q16,None,None,q3,None,None,None
5,q9,None,None,q4,None,None,None
6,None,None,None,q5,None,None,None
7,None,None,None,q6,None,None,None
8,None,None,None,q7,None,None,None
9,None,None,None,q8,None,None,None


In [ ]:
output_filename = 'burns_classification_output.xlsx'
final_df.to_excel(output_filename, index=False)

print(f"Final classification saved to {output_filename}")

# To offer a download link in Colab
from google.colab import files
files.download(output_filename)

Final classification saved to burns_classification_output.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

You can download the Excel file using the link above.